# Sentiment Analysis and Opinion Mining of FIFA 23 Steam Reviews

This notebook is the complete analysis workflow for the Social Media Computing assignment: **Sentiment Analysis and Opinion Mining of FIFA 23 Steam Reviews**.

The main supervised sentiment label is the Steam recommendation field `voted_up`:

- `True` = Positive sentiment / recommended review
- `False` = Negative sentiment / not recommended review

The notebook is written as a reproducible report pipeline. A new user should be able to install the dependencies, place `fifa23_steam_reviews.csv` beside the notebook or in Google Colab, run all cells from top to bottom, and get the cleaned dataset, charts, model results, hypothesis test results, error-analysis samples, and aspect-sentiment outputs.

The workflow covers data loading, cleaning, text preprocessing, EDA, hypothesis testing, TF-IDF feature engineering, traditional supervised ML, transformer sentiment comparison, opinion mining, aspect-based sentiment analysis, final summaries, and exported report files.


## Problem Statement

User reviews on online gaming platforms contain valuable signals about player satisfaction, frustration, and expectations. However, manually reading thousands of reviews is time-consuming and difficult to summarize objectively.

This project applies sentiment analysis and opinion mining techniques to **FIFA 23 Steam reviews** in order to classify review sentiment, identify common opinion words, examine player behavior metadata, and discover which game aspects receive positive or negative feedback.

The main supervised label is `voted_up`, which represents whether a Steam user recommended the game. In this notebook, `voted_up = True` is treated as Positive sentiment and `voted_up = False` is treated as Negative sentiment. The analysis combines exploratory visualization, statistical hypothesis testing, traditional machine learning, transformer-based sentiment prediction, and lightweight aspect-based sentiment analysis to produce outputs that can be used directly in the final report.


## Research Hypotheses

The notebook tests the following hypotheses using cleaned FIFA 23 Steam review data:

**H1: Recent Playtime and Positive Sentiment**

- Alternative hypothesis: Players with higher recent playtime in the last two weeks are more likely to leave positive reviews.
- Null hypothesis: There is no significant difference in recent playtime between positive and negative reviews.

**H2: Number of Games Owned and Negative Sentiment**

- Alternative hypothesis: Players who own more games are more likely to provide negative reviews.
- Null hypothesis: There is no significant difference in the number of games owned between positive and negative reviewers.

**H3: Sentiment Changes Over Time**

- Alternative hypothesis: The proportion of positive and negative reviews changes across review months.
- Null hypothesis: Review sentiment distribution is independent of review month.

**H4: Written Sentiment and Steam Recommendation Agreement**

- Alternative hypothesis: Lexicon-based written sentiment agrees with the Steam `voted_up` recommendation label.
- Null hypothesis: Lexicon-based written sentiment does not meaningfully agree with the Steam `voted_up` recommendation label.


## How to Run or Reuse This Notebook

Use these steps when running the notebook in Google Colab or locally:

1. Keep `requirements.txt` in the same folder as this notebook.
2. The first setup cell installs dependencies with `!pip install -r requirements.txt`.
3. Put `fifa23_steam_reviews.csv` in the same folder as this notebook, upload it to `/content/` in Colab, or place it in one of the Google Drive paths checked by the loader.
4. Select `Runtime > Restart and run all` in Colab.
5. After the notebook finishes, use `outputs/figures/` for PNG charts and `outputs/*.csv` for report tables.

The original CSV is never modified. All derived files are written into the `outputs/` folder.


## 1. Project Setup

This section prepares the runtime environment. It installs dependencies from `requirements.txt`, imports the libraries used throughout the analysis, downloads NLTK resources, sets a fixed random seed, configures chart styling, and creates output folders.

The most important reusable objects created here are `RANDOM_STATE = 42` for reproducibility, `OUTPUT_DIR` and `FIGURE_DIR` for saved artifacts, and `savefig()` for consistent figure export.


In [ ]:
# Install project dependencies.
# If requirements.txt is available, use it so everyone gets the same package list.
# If it is missing, install the core packages required by this notebook.
import subprocess
import sys
from pathlib import Path

requirements_path = Path("requirements.txt")

if requirements_path.exists():
    print("Installing dependencies from requirements.txt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
else:
    print("requirements.txt not found. Installing core notebook dependencies...")
    fallback_packages = [
        "pandas",
        "numpy",
        "matplotlib",
        "seaborn",
        "wordcloud",
        "nltk",
        "scipy",
        "scikit-learn",
        "transformers",
        "torch",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *fallback_packages])

print("Dependency setup complete.")


In [ ]:
# Import analysis, visualization, NLP, statistics, and machine-learning libraries. The fixed random seed and output folders are also configured here.
import os, re, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk import pos_tag

from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)
RANDOM_STATE = 42

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["axes.titleweight"] = "bold"

OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

for resource in ["stopwords", "wordnet", "omw-1.4", "vader_lexicon", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception as exc:
        print(f"NLTK download warning for {resource}: {exc}")

def savefig(filename):
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")

print("Setup complete.")


## 2. Data Loading and Initial Inspection

This section loads the raw Steam reviews and checks whether the dataset matches the expected assignment structure. It displays the shape, column names, data types, first rows, missing values, language counts, sentiment distribution, date ranges, and summary statistics for important metadata columns.

The loader checks `/content`, the current notebook folder, common Google Drive paths, and then manual Colab upload as a fallback.


In [ ]:
# Locate and load the FIFA 23 reviews CSV without hard-coding one machine-specific path.
def find_dataset_path():
    candidates = [
        Path("/content/fifa23_steam_reviews.csv"),
        Path("fifa23_steam_reviews.csv"),
        Path("/content/drive/MyDrive/fifa23_steam_reviews.csv"),
        Path("/content/drive/MyDrive/Colab Notebooks/fifa23_steam_reviews.csv"),
        Path("/content/drive/MyDrive/Colab Notebooks/Social Media Computing Week 2/fifa23_steam_reviews.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        for path in candidates:
            if path.exists():
                return path
    except Exception as exc:
        print(f"Drive lookup skipped or failed: {exc}")
    try:
        from google.colab import files
        print("Please upload fifa23_steam_reviews.csv")
        uploaded = files.upload()
        return Path(next(iter(uploaded.keys())))
    except Exception as exc:
        raise FileNotFoundError("Could not find fifa23_steam_reviews.csv.") from exc

data_path = find_dataset_path()
df = pd.read_csv(data_path)
if "recieved_for_free" in df.columns and "received_for_free" not in df.columns:
    df["received_for_free"] = df["recieved_for_free"]
df["voted_up"] = df["voted_up"].astype(str).str.strip().str.lower().map({"true": True, "false": False})

print(f"Loaded from: {data_path}")
print("Shape:", df.shape)
display(df.head())
display(pd.DataFrame({"dtype": df.dtypes.astype(str)}))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
display(df["voted_up"].map({True: "Positive", False: "Negative"}).value_counts(dropna=False).to_frame("review_count"))

created_dates = pd.to_datetime(df["created"], errors="coerce")
last_played_dates = pd.to_datetime(df["author_last_played"], errors="coerce")
print(f"Created date range: {created_dates.min()} to {created_dates.max()}")
print(f"Author last played date range: {last_played_dates.min()} to {last_played_dates.max()}")

summary_cols = ["votes_up", "comment_count", "author_num_games_owned", "author_num_reviews", "author_playtime_forever", "author_playtime_last_two_weeks", "author_playtime_at_review"]
display(df[summary_cols].describe().T)


## 3. Data Cleaning

This section prepares the dataset for analysis while preserving the original review text.

Rows with missing or empty reviews are removed, exact duplicate review texts are dropped, date columns are parsed, supervised sentiment labels are created from `voted_up`, review length is calculated, and playtime minutes are converted into hours for clearer interpretation.


In [ ]:
# Build a cleaned analysis dataframe while preserving the raw loaded dataframe and original CSV file.
df_clean = df.copy()
df_clean["raw_review"] = df_clean["review"].astype("string").fillna("").str.strip()

before_empty = len(df_clean)
df_clean = df_clean[df_clean["raw_review"].str.len() > 0].copy()
after_empty = len(df_clean)

before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="raw_review", keep="first").copy()
after_dup = len(df_clean)

df_clean["created"] = pd.to_datetime(df_clean["created"], errors="coerce")
df_clean["author_last_played"] = pd.to_datetime(df_clean["author_last_played"], errors="coerce")
df_clean = df_clean[df_clean["voted_up"].notna()].copy()
df_clean["sentiment_binary"] = df_clean["voted_up"].astype(int)
df_clean["sentiment_label"] = df_clean["sentiment_binary"].map({1: "Positive", 0: "Negative"})
df_clean["created_month"] = df_clean["created"].dt.to_period("M").dt.to_timestamp()
df_clean["review_length_words"] = df_clean["raw_review"].str.split().str.len()

for col in ["author_playtime_forever", "author_playtime_last_two_weeks", "author_playtime_at_review"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
    df_clean[f"{col}_hours"] = df_clean[col] / 60

print(f"Rows before empty review drop: {before_empty:,}")
print(f"Rows after empty review drop: {after_empty:,}")
print(f"Exact duplicate review texts removed: {before_dup - after_dup:,}")
print(f"Final cleaned rows before text preprocessing: {len(df_clean):,}")
display(df_clean[["raw_review", "sentiment_label", "created", "created_month", "review_length_words", "author_playtime_last_two_weeks_hours"]].head())


## 4. Text Preprocessing

This section converts raw review text into `clean_review`, which is used for word-frequency analysis, TF-IDF features, opinion mining, and traditional ML models.

The cleaning function lowercases text, removes URLs, removes punctuation/special characters and numbers, tokenizes on whitespace, removes stopwords, and lemmatizes words. Negation and sentiment words such as `not`, `no`, `never`, `bad`, and `good` are kept because removing them can flip meaning.


In [ ]:
# Create the reusable text-cleaning function. Negations and key sentiment words are intentionally preserved.
lemmatizer = WordNetLemmatizer()
keep_words = {"not", "no", "nor", "never", "bad", "good"}
stop_words = (set(stopwords.words("english")) - keep_words).union({"fifa", "fifa23", "steam", "review", "reviews"})

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [t for t in text.split() if t not in stop_words and len(t) > 1]
    return " ".join(lemmatizer.lemmatize(t) for t in tokens)

df_clean["clean_review"] = df_clean["raw_review"].apply(clean_text)
display(df_clean[["raw_review", "clean_review", "sentiment_label"]].sample(5, random_state=RANDOM_STATE))
print("Empty clean reviews:", int((df_clean["clean_review"].str.len() == 0).sum()))


## 5. Exploratory Data Analysis and Visualizations

This section creates report-ready visual evidence about the FIFA 23 review dataset. The charts describe sentiment balance, review activity over time, review length, player metadata differences by sentiment, frequent words, word clouds, and TF-IDF terms.

Every plot has a clear title, axis labels, and a saved PNG file in `outputs/figures/`.


In [ ]:
# Generate the first group of EDA figures: sentiment distribution, monthly trends, length distribution, and metadata-by-sentiment plots.
# 1) Bar chart: count positive and negative Steam recommendations.
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df_clean, x="sentiment_label", order=["Positive", "Negative"], palette=["#4CAF50", "#F44336"])
ax.set(title="FIFA 23 Steam Review Sentiment Distribution", xlabel="Sentiment based on voted_up", ylabel="Number of reviews")
for c in ax.containers: ax.bar_label(c, fmt="%d")
savefig("01_sentiment_distribution_bar.png")

# 2) Pie chart: show the percentage share of each sentiment class.
sentiment_counts = df_clean["sentiment_label"].value_counts().reindex(["Positive", "Negative"])
plt.figure(figsize=(6, 6))
plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct="%1.1f%%", startangle=90, colors=["#4CAF50", "#F44336"])
plt.title("FIFA 23 Steam Review Sentiment Share")
savefig("02_sentiment_distribution_pie.png")

# Create one monthly summary table reused by both time-series plots and H3.
monthly_sentiment = df_clean.dropna(subset=["created_month"]).groupby("created_month").agg(
    review_count=("raw_review", "size"),
    average_sentiment=("sentiment_binary", "mean"),
    positive_percentage=("sentiment_binary", lambda x: x.mean() * 100),
).reset_index()

# 3) Review volume trend: how many reviews were created each month.
plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sentiment, x="created_month", y="review_count", marker="o", color="#3366CC")
plt.title("FIFA 23 Steam Review Count Over Time by Month"); plt.xlabel("Review month"); plt.ylabel("Number of reviews"); plt.xticks(rotation=45)
savefig("03_review_count_over_time_by_month.png")

# 4) Sentiment trend: percentage of reviews that are positive each month.
plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sentiment, x="created_month", y="positive_percentage", marker="o", color="#2E7D32")
plt.title("FIFA 23 Positive Review Percentage Over Time"); plt.xlabel("Review month"); plt.ylabel("Positive reviews (%)"); plt.ylim(0, 100); plt.xticks(rotation=45)
savefig("04_average_sentiment_over_time_by_month.png")

# 5) Review length distribution. The 99th percentile cap prevents a few very long reviews from hiding the main pattern.
plt.figure(figsize=(10, 5))
sns.histplot(df_clean["review_length_words"].clip(upper=df_clean["review_length_words"].quantile(0.99)), bins=40, kde=True, color="#6A5ACD")
plt.title("FIFA 23 Review Length Distribution"); plt.xlabel("Review length in words (capped at 99th percentile)"); plt.ylabel("Number of reviews")
savefig("05_review_length_distribution.png")

# 6) Compare recent playtime distributions for positive and negative reviewers.
plt.figure(figsize=(8, 5))
sns.violinplot(data=df_clean, x="sentiment_label", y="author_playtime_last_two_weeks_hours", order=["Positive", "Negative"], cut=0, inner="quartile", palette=["#4CAF50", "#F44336"])
plt.ylim(0, df_clean["author_playtime_last_two_weeks_hours"].quantile(0.98)); plt.title("FIFA 23 Recent Playtime Hours by Sentiment"); plt.xlabel("Sentiment based on voted_up"); plt.ylabel("Recent playtime in last two weeks (hours)")
savefig("06_recent_playtime_hours_by_sentiment.png")

# 7) Compare number of owned games across sentiment groups.
plt.figure(figsize=(8, 5))
sns.violinplot(data=df_clean, x="sentiment_label", y="author_num_games_owned", order=["Positive", "Negative"], cut=0, inner="quartile", palette=["#4CAF50", "#F44336"])
plt.ylim(0, df_clean["author_num_games_owned"].quantile(0.98)); plt.title("FIFA 23 Number of Games Owned by Sentiment"); plt.xlabel("Sentiment based on voted_up"); plt.ylabel("Number of games owned")
savefig("07_games_owned_by_sentiment.png")


In [ ]:
# Generate text-focused EDA figures: correlation heatmap, frequent words, word clouds, and class-specific TF-IDF terms.
# Select numeric metadata fields plus sentiment so relationships can be inspected in one heatmap.
corr_cols = ["votes_up", "comment_count", "author_num_games_owned", "author_num_reviews", "author_playtime_forever_hours", "author_playtime_last_two_weeks_hours", "author_playtime_at_review_hours", "review_length_words", "sentiment_binary"]
plt.figure(figsize=(11, 8))
sns.heatmap(df_clean[corr_cols].apply(pd.to_numeric, errors="coerce").corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("FIFA 23 Numeric Metadata Correlation Heatmap")
savefig("08_numeric_metadata_correlation_heatmap.png")

# Flatten all cleaned reviews into tokens and count the most common words.
all_tokens = " ".join(df_clean["clean_review"]).split()
top_words_df = pd.DataFrame(Counter(all_tokens).most_common(20), columns=["word", "count"])
words_to_plot = top_words_df["word"].tolist()
counts_to_plot = top_words_df["count"].tolist()
plt.figure(figsize=(11, 6))
sns.barplot(data=top_words_df, x="count", y="word", palette="viridis")
plt.title("Top 20 Frequent Words in FIFA 23 Steam Reviews"); plt.xlabel("Frequency"); plt.ylabel("Word")
savefig("09_top_20_frequent_words_overall.png")
display(top_words_df)

# Reusable word-cloud helper keeps positive and negative word cloud code consistent.
def plot_wordcloud(text, title, filename):
    plt.figure(figsize=(11, 6))
    if str(text).strip():
        cloud = WordCloud(width=1200, height=650, background_color="white", colormap="viridis", random_state=RANDOM_STATE).generate(text)
        plt.imshow(cloud, interpolation="bilinear")
    else:
        plt.text(0.5, 0.5, "No words available", ha="center", va="center", fontsize=16)
    plt.axis("off"); plt.title(title); savefig(filename)

plot_wordcloud(" ".join(df_clean.loc[df_clean["sentiment_label"] == "Positive", "clean_review"]), "FIFA 23 Positive Review Word Cloud", "10_positive_word_cloud.png")
plot_wordcloud(" ".join(df_clean.loc[df_clean["sentiment_label"] == "Negative", "clean_review"]), "FIFA 23 Negative Review Word Cloud", "11_negative_word_cloud.png")

# Fit an EDA-only TF-IDF vectorizer to compare terms that are prominent in each sentiment class.
eda_tfidf = TfidfVectorizer(max_features=5000, min_df=2, max_df=0.9, ngram_range=(1, 2))
eda_mat = eda_tfidf.fit_transform(df_clean["clean_review"])
features = np.array(eda_tfidf.get_feature_names_out())
# Average TF-IDF scores separately for positive and negative reviews.
pos_scores = np.asarray(eda_mat[df_clean["sentiment_binary"].values == 1].mean(axis=0)).ravel()
neg_scores = np.asarray(eda_mat[df_clean["sentiment_binary"].values == 0].mean(axis=0)).ravel()
top_positive_terms = pd.DataFrame({"term": features[np.argsort(pos_scores)[-20:][::-1]], "tfidf_score": np.sort(pos_scores)[-20:][::-1]})
top_negative_terms = pd.DataFrame({"term": features[np.argsort(neg_scores)[-20:][::-1]], "tfidf_score": np.sort(neg_scores)[-20:][::-1]})

plt.figure(figsize=(11, 6)); sns.barplot(data=top_positive_terms, x="tfidf_score", y="term", palette="Greens_r")
plt.title("Top Positive TF-IDF Terms in FIFA 23 Reviews"); plt.xlabel("Mean TF-IDF score in positive reviews"); plt.ylabel("Term")
savefig("12_top_positive_tfidf_terms.png")

plt.figure(figsize=(11, 6)); sns.barplot(data=top_negative_terms, x="tfidf_score", y="term", palette="Reds_r")
plt.title("Top Negative TF-IDF Terms in FIFA 23 Reviews"); plt.xlabel("Mean TF-IDF score in negative reviews"); plt.ylabel("Term")
savefig("13_top_negative_tfidf_terms.png")
display(top_positive_terms); display(top_negative_terms)


## 6. Hypothesis Testing

This section tests the report hypotheses using statistical methods that fit the data types.

H1 and H2 compare skewed numeric metadata between positive and negative reviews using Mann-Whitney U tests. H3 tests whether sentiment distribution changes by month using a chi-square test and a trend chart. H4 compares written review polarity with the Steam `voted_up` label using VADER lexicon sentiment.


In [ ]:
# Run the four report hypothesis checks and store the results in a dataframe for export.
hypothesis_results = []
def add_hypothesis_result(hypothesis, test, statistic, p_value, interpretation):
    hypothesis_results.append({"Hypothesis": hypothesis, "Test": test, "Statistic": statistic, "p_value": p_value, "Interpretation": interpretation})

def mann_whitney(column, hypothesis):
    pos = df_clean.loc[df_clean["sentiment_label"] == "Positive", column].dropna()
    neg = df_clean.loc[df_clean["sentiment_label"] == "Negative", column].dropna()
    stat, p = mannwhitneyu(pos, neg, alternative="two-sided")
    interp = f"p={p:.4g}; Positive median={pos.median():.2f}, Negative median={neg.median():.2f}. " + ("Significant difference detected." if p < 0.05 else "No statistically significant difference detected.")
    add_hypothesis_result(hypothesis, "Mann-Whitney U test", stat, p, interp)
    print(hypothesis, "\n", interp, "\n")

mann_whitney("author_playtime_last_two_weeks_hours", "H1: Players with higher recent playtime are more likely to leave positive reviews")
mann_whitney("author_num_games_owned", "H2: Players who own more games are more likely to provide negative reviews")

month_table = pd.crosstab(df_clean["created_month"], df_clean["sentiment_label"])
if month_table.shape[0] > 1 and month_table.shape[1] > 1:
    chi2_stat, chi2_p, _, _ = chi2_contingency(month_table)
    h3_interp = f"p={chi2_p:.4g}; " + ("sentiment distribution changes significantly over time." if chi2_p < 0.05 else "no significant month-by-sentiment association detected.")
else:
    chi2_stat, chi2_p, h3_interp = np.nan, np.nan, "Not enough monthly groups for chi-square testing."
add_hypothesis_result("H3: Sentiment changes over time", "Chi-square test of month vs sentiment", chi2_stat, chi2_p, h3_interp)
print("H3:", h3_interp)

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sentiment, x="created_month", y="positive_percentage", marker="o", color="#00695C")
plt.title("H3: FIFA 23 Positive Review Percentage by Month"); plt.xlabel("Review month"); plt.ylabel("Positive reviews (%)"); plt.ylim(0, 100); plt.xticks(rotation=45)
savefig("14_h3_positive_review_percentage_by_month.png")

sia = SentimentIntensityAnalyzer()
df_clean["vader_compound"] = df_clean["raw_review"].apply(lambda t: sia.polarity_scores(str(t))["compound"])
df_clean["vader_label_detailed"] = np.select([df_clean["vader_compound"] >= 0.05, df_clean["vader_compound"] <= -0.05], ["Positive", "Negative"], default="Neutral")
df_clean["vader_binary_label"] = np.where(df_clean["vader_compound"] >= 0, "Positive", "Negative")
agreement = (df_clean["vader_binary_label"] == df_clean["sentiment_label"]).mean()
h4_interp = f"VADER binary sentiment agrees with voted_up for {agreement:.2%} of cleaned reviews. Mismatches may indicate sarcasm, slang, short text, or mixed opinions."
add_hypothesis_result("H4: Written sentiment agrees with Steam voted_up recommendation", "VADER vs voted_up agreement", agreement, np.nan, h4_interp)
print("H4:", h4_interp)

cm = confusion_matrix(df_clean["sentiment_label"], df_clean["vader_binary_label"], labels=["Positive", "Negative"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["VADER Positive", "VADER Negative"], yticklabels=["Actual Positive", "Actual Negative"])
plt.title("H4: VADER Sentiment vs Steam voted_up"); plt.xlabel("VADER predicted sentiment"); plt.ylabel("Steam voted_up sentiment")
savefig("15_h4_vader_vs_voted_up_confusion_matrix.png")

hypothesis_results_df = pd.DataFrame(hypothesis_results)
display(hypothesis_results_df)
display(df_clean.loc[df_clean["vader_binary_label"] != df_clean["sentiment_label"], ["raw_review", "sentiment_label", "vader_binary_label", "vader_compound"]].head(10))


## 7. Feature Engineering

This section converts cleaned review text into TF-IDF features for supervised machine learning. TF-IDF gives higher weight to terms that are important in a review but not too common across all reviews.

The split is stratified 80/20 so the train and test sets keep a similar positive/negative class distribution. The vectorizer uses unigrams and bigrams to capture both single words and short phrases.


In [ ]:
# Convert cleaned text into TF-IDF features and create a stratified 80/20 train-test split.
X = df_clean["clean_review"]
y = df_clean["sentiment_binary"]
X_train_text, X_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2, max_df=0.9)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)
display(y_train.map({1: "Positive", 0: "Negative"}).value_counts(normalize=True).to_frame("training_share"))


## 8. Traditional Machine Learning Models

This section trains supervised models that learn from TF-IDF review features and the `voted_up` sentiment label.

Models include Multinomial Naive Bayes, Logistic Regression, Linear SVM, and Random Forest. Each model is evaluated with accuracy, precision, recall, F1-score, a classification report, and a confusion matrix.


In [ ]:
# Train and evaluate the traditional supervised ML models using the same TF-IDF features and test set.
# Define models in one dictionary so the same training/evaluation loop can be reused.
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=1),
    "Linear SVM": LinearSVC(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=60, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=1, class_weight="balanced_subsample"),
}
trained_models, model_predictions, rows = {}, {}, []
# Train each model, predict on the same test set, and store comparable metrics.
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_test_tfidf)
    trained_models[name], model_predictions[name] = model, pred
    # Positive class is sentiment_binary=1, so precision/recall/F1 describe positive-review detection.
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred), "Precision": precision_score(y_test, pred, zero_division=0), "Recall": recall_score(y_test, pred, zero_division=0), "F1-score": f1_score(y_test, pred, zero_division=0)})
    print(classification_report(y_test, pred, target_names=["Negative", "Positive"], zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=[1, 0])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Predicted Positive", "Predicted Negative"], yticklabels=["Actual Positive", "Actual Negative"])
    plt.title(f"FIFA 23 Confusion Matrix: {name}"); plt.xlabel("Predicted sentiment"); plt.ylabel("Actual sentiment")
    savefig(f"16_confusion_matrix_{name.lower().replace(' ', '_').replace('-', '_')}.png")

model_results_df = pd.DataFrame(rows).sort_values("F1-score", ascending=False).reset_index(drop=True)
display(model_results_df)

# Convert the wide metrics table into long format for a grouped bar chart.
long_results = model_results_df.melt(id_vars="Model", value_vars=["Accuracy", "Precision", "Recall", "F1-score"], var_name="Metric", value_name="Score")
plt.figure(figsize=(11, 6))
sns.barplot(data=long_results, x="Model", y="Score", hue="Metric")
plt.title("FIFA 23 Traditional ML Model Performance Comparison"); plt.xlabel("Model"); plt.ylabel("Score"); plt.ylim(0, 1); plt.xticks(rotation=20, ha="right"); plt.legend(title="Metric", loc="lower right")
savefig("17_traditional_ml_model_performance_comparison.png")

# Cross-validation estimates whether model performance is stable across different folds.
# n_jobs=1 is used for portability in restricted Windows/Colab-style environments.
cv_results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, model in {k: models[k] for k in ["Multinomial Naive Bayes", "Logistic Regression", "Linear SVM"]}.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=10000, min_df=2, max_df=0.9)), ("model", model)])
    scores = cross_val_score(pipe, X, y, cv=skf, scoring="f1", n_jobs=1)
    cv_results.append({"Model": name, "Mean_CV_F1": scores.mean(), "Std_CV_F1": scores.std()})
    print(f"{name}: mean CV F1={scores.mean():.4f}, std={scores.std():.4f}")
cv_results_df = pd.DataFrame(cv_results)
display(cv_results_df)


## 9. Error Analysis

This section inspects cases where the best traditional model disagrees with the actual `voted_up` label. False positives are negative reviews predicted as positive, while false negatives are positive reviews predicted as negative.

These examples help explain model limitations such as sarcasm, short reviews, slang, mixed opinions, and football or gaming-specific vocabulary.


In [ ]:
# Inspect false positives and false negatives from the best traditional model.
best_traditional_model_name = model_results_df.iloc[0]["Model"]
best_pred = model_predictions[best_traditional_model_name]
print(f"Best traditional model by F1-score: {best_traditional_model_name}")

test_rows = df_clean.loc[X_test_text.index].copy()
test_rows["actual_sentiment"] = y_test.map({1: "Positive", 0: "Negative"}).values
test_rows["predicted_sentiment"] = pd.Series(best_pred, index=y_test.index).map({1: "Positive", 0: "Negative"}).values

fp = test_rows[(test_rows["actual_sentiment"] == "Negative") & (test_rows["predicted_sentiment"] == "Positive")]
fn = test_rows[(test_rows["actual_sentiment"] == "Positive") & (test_rows["predicted_sentiment"] == "Negative")]
error_analysis_df = pd.concat([
    fp[["raw_review", "actual_sentiment", "predicted_sentiment", "review_length_words", "author_playtime_forever_hours"]].head(10).assign(error_type="False Positive"),
    fn[["raw_review", "actual_sentiment", "predicted_sentiment", "review_length_words", "author_playtime_forever_hours"]].head(10).assign(error_type="False Negative"),
], ignore_index=True).rename(columns={"raw_review": "review", "author_playtime_forever_hours": "playtime_hours"})
display(error_analysis_df)


Common error causes include sarcasm, short reviews, slang, mixed opinions, and football/gaming-specific terms whose sentiment depends on player context.


## 10. Deep Learning / Transformer Model

This section adds the required deep-learning or transformer component using a pretrained Hugging Face sentiment pipeline.

To keep the notebook practical for Colab, the transformer is not fine-tuned. Instead, DistilBERT predicts sentiment for a stratified sample of reviews and the predictions are compared with the Steam `voted_up` label. This gives a fair benchmark against traditional ML while avoiding a long training process.

The enhanced analysis also checks confidence and mismatch examples. This is useful for the report because pretrained sentiment models may misunderstand gaming slang, sarcasm, or recommendation labels that do not perfectly match written sentiment.


In [ ]:
# Run a practical transformer comparison on a stratified sample so the section remains Colab-friendly.
# Limit the sample size so the transformer section is useful but still realistic on free Colab/CPU sessions.
TRANSFORMER_SAMPLE_SIZE = min(2000, len(df_clean))
transformer_results_df = pd.DataFrame(columns=["Model", "Accuracy", "Precision", "Recall", "F1-score"])
transformer_predictions_df = pd.DataFrame()
transformer_error_examples_df = pd.DataFrame()
transformer_confidence_summary_df = pd.DataFrame()

try:
    import torch
    from transformers import pipeline

    # Stratified sampling keeps the same positive/negative balance as the full cleaned dataset.
    if TRANSFORMER_SAMPLE_SIZE < len(df_clean):
        _, transformer_sample = train_test_split(
            df_clean,
            test_size=TRANSFORMER_SAMPLE_SIZE,
            stratify=df_clean["sentiment_binary"],
            random_state=RANDOM_STATE,
        )
    else:
        transformer_sample = df_clean.copy()

    # Hugging Face uses device=0 for GPU and device=-1 for CPU.
    device = 0 if torch.cuda.is_available() else -1
    print(f"Running DistilBERT sentiment pipeline on {len(transformer_sample):,} reviews with device={device}")
    sentiment_pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device)

    # Pretrained transformers expect natural text, so raw_review is better than clean_review here.
    outputs = sentiment_pipe(
        transformer_sample["raw_review"].astype(str).str.slice(0, 2000).tolist(),
        batch_size=32,
        truncation=True,
        max_length=512,
    )

    transformer_sample = transformer_sample.copy()
    transformer_sample["transformer_raw_label"] = [item["label"] for item in outputs]
    transformer_sample["transformer_score"] = [item["score"] for item in outputs]
    transformer_sample["transformer_prediction"] = transformer_sample["transformer_raw_label"].str.upper().map({"POSITIVE": 1, "NEGATIVE": 0})
    transformer_sample = transformer_sample.dropna(subset=["transformer_prediction"]).copy()
    transformer_sample["transformer_prediction"] = transformer_sample["transformer_prediction"].astype(int)
    transformer_sample["transformer_sentiment"] = transformer_sample["transformer_prediction"].map({1: "Positive", 0: "Negative"})
    transformer_sample["transformer_matches_voted_up"] = transformer_sample["transformer_prediction"] == transformer_sample["sentiment_binary"]
    transformer_sample["confidence_band"] = pd.cut(
        transformer_sample["transformer_score"],
        bins=[0, 0.70, 0.85, 1.0],
        labels=["Low/Medium", "High", "Very High"],
        include_lowest=True,
    )

    # Evaluate transformer predictions against the same voted_up label used by the supervised ML models.
    t_y = transformer_sample["sentiment_binary"]
    t_pred = transformer_sample["transformer_prediction"]
    transformer_results_df = pd.DataFrame([{
        "Model": "Transformer DistilBERT SST-2",
        "Accuracy": accuracy_score(t_y, t_pred),
        "Precision": precision_score(t_y, t_pred, zero_division=0),
        "Recall": recall_score(t_y, t_pred, zero_division=0),
        "F1-score": f1_score(t_y, t_pred, zero_division=0),
    }])
    print(classification_report(t_y, t_pred, target_names=["Negative", "Positive"], zero_division=0))

    cm = confusion_matrix(t_y, t_pred, labels=[1, 0])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=["Predicted Positive", "Predicted Negative"], yticklabels=["Actual Positive", "Actual Negative"])
    plt.title("FIFA 23 Transformer Confusion Matrix")
    plt.xlabel("Predicted sentiment")
    plt.ylabel("Actual sentiment")
    savefig("18_transformer_confusion_matrix.png")

    # Confidence analysis helps explain when the transformer is confidently wrong.
    transformer_confidence_summary_df = (
        transformer_sample.groupby(["confidence_band", "transformer_matches_voted_up"], observed=False)
        .size()
        .reset_index(name="review_count")
    )
    display(transformer_confidence_summary_df)

    plt.figure(figsize=(8, 5))
    sns.boxplot(data=transformer_sample, x="transformer_matches_voted_up", y="transformer_score", palette="Set2")
    plt.title("FIFA 23 Transformer Confidence by Correctness")
    plt.xlabel("Prediction matches voted_up label")
    plt.ylabel("Transformer confidence score")
    savefig("24_transformer_confidence_by_correctness.png")

    # Mismatch examples are useful in the report discussion section.
    transformer_error_examples_df = (
        transformer_sample.loc[
            ~transformer_sample["transformer_matches_voted_up"],
            ["raw_review", "sentiment_label", "transformer_sentiment", "transformer_score", "review_length_words"],
        ]
        .sort_values("transformer_score", ascending=False)
        .head(15)
        .rename(columns={"raw_review": "review"})
    )
    display(transformer_error_examples_df)

    transformer_predictions_df = transformer_sample[[
        "raw_review",
        "sentiment_label",
        "transformer_raw_label",
        "transformer_score",
        "transformer_prediction",
        "transformer_sentiment",
        "transformer_matches_voted_up",
        "confidence_band",
    ]].copy()
except Exception as exc:
    print("Transformer pipeline could not be completed in this runtime.")
    print("The setup cell includes install commands. Rerun this section with internet/model access if needed.")
    print(f"Error: {exc}")

# Add transformer metrics to the same comparison table used for traditional models.
combined_model_results_df = pd.concat([model_results_df, transformer_results_df], ignore_index=True).dropna(subset=["F1-score"])
plt.figure(figsize=(12, 6))
sns.barplot(data=combined_model_results_df.sort_values("F1-score", ascending=False), x="Model", y="F1-score", palette="mako")
plt.title("FIFA 23 Traditional ML vs Transformer F1-score Comparison")
plt.xlabel("Model")
plt.ylabel("F1-score")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
savefig("19_traditional_ml_vs_transformer_f1_comparison.png")
display(combined_model_results_df.sort_values("F1-score", ascending=False))


## 11. Opinion Mining

This section extracts common opinion words and simple opinion phrases from positive and negative reviews.

The approach stays lightweight: POS tagging is used to identify adjectives, and adjective-noun pairs are used as simple opinion phrases or opinion targets. This is easier to explain than full dependency parsing, but it still gives more insight than a basic word cloud because it shows what players describe positively or negatively.


In [ ]:
# Extract common adjective/opinion words and simple adjective-noun opinion phrases.
def get_pos_tags(text):
    '''Tokenize a cleaned review and return NLTK POS tags.'''
    tokens = [token for token in str(text).split() if len(token) > 2]
    if not tokens:
        return []
    try:
        return pos_tag(tokens)
    except Exception:
        return [(token, "") for token in tokens]


def top_adjectives(text_series, top_n=20):
    '''Return frequent adjectives, which often carry opinion in review text.'''
    counter = Counter()
    for text in text_series.astype(str):
        tagged_tokens = get_pos_tags(text)
        counter.update([word for word, tag in tagged_tokens if tag.startswith("JJ")])
    return pd.DataFrame(counter.most_common(top_n), columns=["opinion_word", "count"])


def top_opinion_phrases(text_series, top_n=20):
    '''Return simple adjective-noun pairs such as bad gameplay or good graphics.'''
    phrase_counter = Counter()
    for text in text_series.astype(str):
        tagged_tokens = get_pos_tags(text)
        for (word_1, tag_1), (word_2, tag_2) in zip(tagged_tokens, tagged_tokens[1:]):
            if tag_1.startswith("JJ") and (tag_2.startswith("NN") or tag_2.startswith("VB")):
                phrase_counter[f"{word_1} {word_2}"] += 1
    return pd.DataFrame(phrase_counter.most_common(top_n), columns=["opinion_phrase", "count"])


positive_reviews_clean = df_clean.loc[df_clean["sentiment_label"] == "Positive", "clean_review"]
negative_reviews_clean = df_clean.loc[df_clean["sentiment_label"] == "Negative", "clean_review"]

# Extract opinion vocabulary separately so positive and negative language can be compared.
positive_opinion_words = top_adjectives(positive_reviews_clean)
negative_opinion_words = top_adjectives(negative_reviews_clean)
positive_opinion_phrases = top_opinion_phrases(positive_reviews_clean)
negative_opinion_phrases = top_opinion_phrases(negative_reviews_clean)

plt.figure(figsize=(10, 6))
sns.barplot(data=positive_opinion_words, x="count", y="opinion_word", palette="Greens_r")
plt.title("Top Opinion Words in Positive FIFA 23 Reviews")
plt.xlabel("Frequency")
plt.ylabel("Opinion word")
savefig("20_top_positive_opinion_words.png")

plt.figure(figsize=(10, 6))
sns.barplot(data=negative_opinion_words, x="count", y="opinion_word", palette="Reds_r")
plt.title("Top Opinion Words in Negative FIFA 23 Reviews")
plt.xlabel("Frequency")
plt.ylabel("Opinion word")
savefig("21_top_negative_opinion_words.png")

plt.figure(figsize=(10, 6))
sns.barplot(data=positive_opinion_phrases, x="count", y="opinion_phrase", palette="Greens_r")
plt.title("Top Opinion Phrases in Positive FIFA 23 Reviews")
plt.xlabel("Frequency")
plt.ylabel("Opinion phrase")
savefig("25_top_positive_opinion_phrases.png")

plt.figure(figsize=(10, 6))
sns.barplot(data=negative_opinion_phrases, x="count", y="opinion_phrase", palette="Reds_r")
plt.title("Top Opinion Phrases in Negative FIFA 23 Reviews")
plt.xlabel("Frequency")
plt.ylabel("Opinion phrase")
savefig("26_top_negative_opinion_phrases.png")

print("Positive opinion words:")
display(positive_opinion_words)
print("Negative opinion words:")
display(negative_opinion_words)
print("Positive opinion phrases:")
display(positive_opinion_phrases)
print("Negative opinion phrases:")
display(negative_opinion_phrases)

# VADER extremes provide qualitative examples of strongly written sentiment.
strong_positive_examples_df = df_clean.sort_values("vader_compound", ascending=False)[["raw_review", "sentiment_label", "vader_compound"]].head(10)
strong_negative_examples_df = df_clean.sort_values("vader_compound", ascending=True)[["raw_review", "sentiment_label", "vader_compound"]].head(10)
display(strong_positive_examples_df)
display(strong_negative_examples_df)


## 12. Aspect-Based Sentiment Analysis

This section performs lightweight aspect-based sentiment analysis suitable for the assignment report.

Instead of training a complex ABSA model, it uses transparent FIFA/game aspect dictionaries. A review can mention multiple aspects, such as Gameplay and Servers/Online. Each aspect mention is summarized using both the Steam recommendation label (`voted_up`) and VADER written-text polarity. This makes the ABSA section stronger while keeping the method simple and explainable.


In [ ]:
# Detect predefined FIFA/game aspects and summarize positive vs negative sentiment for each aspect.
# Each aspect has a small keyword dictionary that can be reviewed and modified by future users.
aspects = {
    "Gameplay": ["gameplay", "mechanic", "control", "pace", "script", "scripting", "match", "player", "ball"],
    "Graphics": ["graphics", "visual", "animation", "realistic", "face", "stadium"],
    "Performance/Bugs": ["bug", "crash", "lag", "fps", "freeze", "error", "glitch", "stutter"],
    "Servers/Online": ["server", "online", "connection", "disconnect", "ping", "multiplayer"],
    "Career/Content": ["career", "mode", "ultimate", "fut", "content", "team", "pack"],
    "Price/Monetization": ["price", "money", "refund", "pay", "microtransaction", "ea"],
}


def detect_aspects_with_keywords(text):
    '''Return aspect and matched keyword pairs for a review.'''
    text = str(text).lower()
    found = []
    for aspect, keywords in aspects.items():
        for keyword in keywords:
            if re.search(r"\b" + re.escape(keyword) + r"s?\b", text):
                found.append((aspect, keyword))
                break
    return found


# Expand from one row per review to one row per detected aspect mention.
aspect_rows = []
for idx, row in df_clean.iterrows():
    for aspect, matched_keyword in detect_aspects_with_keywords(row["raw_review"]):
        aspect_rows.append({
            "review_index": idx,
            "aspect": aspect,
            "matched_keyword": matched_keyword,
            "sentiment_label": row["sentiment_label"],
            "sentiment_binary": row["sentiment_binary"],
            "vader_compound": row["vader_compound"],
            "vader_binary_label": row["vader_binary_label"],
            "raw_review": row["raw_review"],
        })

aspect_mentions_df = pd.DataFrame(aspect_rows)
print(f"Total aspect mentions detected: {len(aspect_mentions_df):,}")
display(aspect_mentions_df.head())

# Summarize every aspect using both Steam recommendation sentiment and written VADER polarity.
if aspect_mentions_df.empty:
    aspect_sentiment = pd.DataFrame({
        "aspect": list(aspects),
        "positive_count": 0,
        "negative_count": 0,
        "total_mentions": 0,
        "positive_percentage": np.nan,
        "negative_percentage": np.nan,
        "avg_vader_compound": np.nan,
        "vader_positive_count": 0,
        "vader_negative_count": 0,
        "vader_positive_percentage": np.nan,
    })
else:
    steam_counts = pd.crosstab(aspect_mentions_df["aspect"], aspect_mentions_df["sentiment_label"]).reindex(aspects.keys()).fillna(0)
    vader_counts = pd.crosstab(aspect_mentions_df["aspect"], aspect_mentions_df["vader_binary_label"]).reindex(aspects.keys()).fillna(0)
    for label in ["Positive", "Negative"]:
        if label not in steam_counts.columns:
            steam_counts[label] = 0
        if label not in vader_counts.columns:
            vader_counts[label] = 0

    aspect_sentiment = steam_counts[["Positive", "Negative"]].astype(int).reset_index()
    aspect_sentiment.columns = ["aspect", "positive_count", "negative_count"]
    aspect_sentiment["total_mentions"] = aspect_sentiment["positive_count"] + aspect_sentiment["negative_count"]
    aspect_sentiment["positive_percentage"] = np.where(
        aspect_sentiment["total_mentions"] > 0,
        aspect_sentiment["positive_count"] / aspect_sentiment["total_mentions"] * 100,
        np.nan,
    )
    aspect_sentiment["negative_percentage"] = np.where(
        aspect_sentiment["total_mentions"] > 0,
        aspect_sentiment["negative_count"] / aspect_sentiment["total_mentions"] * 100,
        np.nan,
    )

    vader_summary = aspect_mentions_df.groupby("aspect")["vader_compound"].mean().reindex(aspects.keys()).rename("avg_vader_compound").reset_index()
    vader_counts_df = vader_counts[["Positive", "Negative"]].astype(int).reset_index()
    vader_counts_df.columns = ["aspect", "vader_positive_count", "vader_negative_count"]
    aspect_sentiment = aspect_sentiment.merge(vader_summary, on="aspect", how="left").merge(vader_counts_df, on="aspect", how="left")
    aspect_sentiment["vader_positive_percentage"] = np.where(
        aspect_sentiment["total_mentions"] > 0,
        aspect_sentiment["vader_positive_count"] / aspect_sentiment["total_mentions"] * 100,
        np.nan,
    )

display(aspect_sentiment)

ax = aspect_sentiment.set_index("aspect")[["positive_count", "negative_count"]].plot(kind="bar", stacked=True, figsize=(11, 6), color=["#4CAF50", "#F44336"])
ax.set_title("FIFA 23 Aspect-Based Sentiment Counts")
ax.set_xlabel("Aspect")
ax.set_ylabel("Number of aspect mentions")
ax.legend(["Positive", "Negative"], title="Steam voted_up sentiment")
plt.xticks(rotation=25, ha="right")
savefig("22_aspect_based_sentiment_stacked_bar.png")

plt.figure(figsize=(8, 5))
sns.heatmap(aspect_sentiment.set_index("aspect")[["positive_count", "negative_count"]], annot=True, fmt="d", cmap="YlGnBu")
plt.title("FIFA 23 Aspect vs Steam Sentiment Heatmap")
plt.xlabel("Sentiment count")
plt.ylabel("Aspect")
savefig("23_aspect_sentiment_heatmap.png")

plt.figure(figsize=(10, 5))
sns.barplot(data=aspect_sentiment.sort_values("avg_vader_compound"), x="avg_vader_compound", y="aspect", palette="coolwarm")
plt.axvline(0, color="black", linewidth=1)
plt.title("FIFA 23 Average Written Polarity by Aspect")
plt.xlabel("Average VADER compound score")
plt.ylabel("Aspect")
savefig("27_aspect_average_vader_polarity.png")

plt.figure(figsize=(9, 5))
sns.heatmap(
    aspect_sentiment.set_index("aspect")[["positive_percentage", "negative_percentage", "vader_positive_percentage"]],
    annot=True,
    fmt=".1f",
    cmap="RdYlGn",
    center=50,
)
plt.title("FIFA 23 Aspect Sentiment Percentages")
plt.xlabel("Metric")
plt.ylabel("Aspect")
savefig("28_aspect_sentiment_percentage_heatmap.png")

# Save example reviews so the ABSA table can be supported with real review evidence.
aspect_examples = []
if not aspect_mentions_df.empty:
    for aspect in aspects:
        examples = aspect_mentions_df[aspect_mentions_df["aspect"] == aspect].copy()
        examples = examples.sort_values(["sentiment_label", "vader_compound"], ascending=[True, True])
        for _, row in examples.groupby("sentiment_label", group_keys=False).head(2).iterrows():
            aspect_examples.append({
                "aspect": aspect,
                "matched_keyword": row["matched_keyword"],
                "sentiment_label": row["sentiment_label"],
                "vader_compound": row["vader_compound"],
                "review": row["raw_review"],
            })
aspect_examples_df = pd.DataFrame(aspect_examples)
display(aspect_examples_df)


## 13. Final Results Summary

This section prints a concise report summary after all analysis steps have run. It gathers cleaned dataset size, sentiment distribution, best traditional model, transformer result, hypothesis findings, important terms, negative aspects, and a developer recommendation.


In [ ]:
# Collect the most important final values into a report-ready printed summary.
positive_count = int((df_clean["sentiment_label"] == "Positive").sum())
negative_count = int((df_clean["sentiment_label"] == "Negative").sum())
best_f1 = float(model_results_df.iloc[0]["F1-score"])

if not transformer_results_df.empty and transformer_results_df["F1-score"].notna().any():
    transformer_summary = f"{transformer_results_df.iloc[0]['Model']} F1-score = {transformer_results_df.iloc[0]['F1-score']:.4f}"
else:
    transformer_summary = "Transformer results unavailable in this runtime; rerun with internet/model access."

main_negative_aspects = aspect_sentiment[aspect_sentiment["total_mentions"] > 0].sort_values(["negative_percentage", "negative_count"], ascending=False).head(3)["aspect"].tolist()
recommendation = f"Prioritize improvements in {', '.join(main_negative_aspects)} because these aspects show the strongest negative sentiment signals." if main_negative_aspects else "Prioritize the aspects with the highest negative counts after rerunning aspect extraction."

print("FINAL RESULTS SUMMARY")
print("=" * 70)
print(f"Final dataset size after cleaning: {len(df_clean):,} reviews")
print(f"Positive reviews: {positive_count:,} ({positive_count / len(df_clean):.2%})")
print(f"Negative reviews: {negative_count:,} ({negative_count / len(df_clean):.2%})")
print(f"Best traditional model and F1-score: {best_traditional_model_name}, {best_f1:.4f}")
print(f"Transformer performance: {transformer_summary}")
print("\nKey hypothesis findings:")
for _, row in hypothesis_results_df.iterrows():
    print(f"- {row['Hypothesis']}: {row['Interpretation']}")
print("\nTop positive TF-IDF terms:", ", ".join(top_positive_terms["term"].head(10)))
print("Top negative TF-IDF terms:", ", ".join(top_negative_terms["term"].head(10)))
print("Top positive opinion phrases:", ", ".join(positive_opinion_phrases["opinion_phrase"].head(5)))
print("Top negative opinion phrases:", ", ".join(negative_opinion_phrases["opinion_phrase"].head(5)))
print("Main aspects with highest negative sentiment:", ", ".join(main_negative_aspects) if main_negative_aspects else "No aspect mentions detected.")
print("Main recommendation for game developers:", recommendation)


## 14. Export Outputs

This final section saves all reusable analysis outputs. These files are intended for direct insertion or reference in the final report, including model metrics, hypothesis results, aspect sentiment results, error-analysis examples, transformer sample predictions if available, and the cleaned review dataset.


In [ ]:
# Export all generated tables and cleaned data for report use.
# Model and statistical result tables.
model_results_df.to_csv(OUTPUT_DIR / "model_results.csv", index=False)
combined_model_results_df.to_csv(OUTPUT_DIR / "combined_model_results.csv", index=False)
hypothesis_results_df.to_csv(OUTPUT_DIR / "hypothesis_results.csv", index=False)
aspect_sentiment.to_csv(OUTPUT_DIR / "aspect_sentiment_results.csv", index=False)
error_analysis_df.to_csv(OUTPUT_DIR / "sample_error_analysis.csv", index=False)

# Opinion mining outputs.
positive_opinion_words.to_csv(OUTPUT_DIR / "positive_opinion_words.csv", index=False)
negative_opinion_words.to_csv(OUTPUT_DIR / "negative_opinion_words.csv", index=False)
positive_opinion_phrases.to_csv(OUTPUT_DIR / "positive_opinion_phrases.csv", index=False)
negative_opinion_phrases.to_csv(OUTPUT_DIR / "negative_opinion_phrases.csv", index=False)
strong_positive_examples_df.to_csv(OUTPUT_DIR / "strong_positive_examples.csv", index=False)
strong_negative_examples_df.to_csv(OUTPUT_DIR / "strong_negative_examples.csv", index=False)

# Cleaned dataset with both raw_review and clean_review for future analysis.
df_clean.to_csv(OUTPUT_DIR / "final_cleaned_fifa23_reviews.csv", index=False)
cv_results_df.to_csv(OUTPUT_DIR / "cross_validation_results.csv", index=False)

# Optional outputs are written only when their sections ran successfully.
if not transformer_predictions_df.empty:
    transformer_predictions_df.to_csv(OUTPUT_DIR / "transformer_sample_predictions.csv", index=False)
if not transformer_error_examples_df.empty:
    transformer_error_examples_df.to_csv(OUTPUT_DIR / "transformer_error_examples.csv", index=False)
if not transformer_confidence_summary_df.empty:
    transformer_confidence_summary_df.to_csv(OUTPUT_DIR / "transformer_confidence_summary.csv", index=False)
if not aspect_mentions_df.empty:
    aspect_mentions_df.to_csv(OUTPUT_DIR / "aspect_mentions.csv", index=False)
if not aspect_examples_df.empty:
    aspect_examples_df.to_csv(OUTPUT_DIR / "aspect_example_reviews.csv", index=False)

print("Export complete. Output CSV files:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"- {path}")
print("\nFigure files:")
for path in sorted(FIGURE_DIR.glob("*.png")):
    print(f"- {path}")


## Notes for Report Writing

When writing the final report, use the exported tables and figures as evidence rather than manually copying values from intermediate cells.

Recommended mapping:

- Sentiment distribution and EDA: figure files `01` to `13`.
- Hypothesis discussion: `hypothesis_results.csv` and the H3/H4 figures.
- Model comparison: `model_results.csv`, `cross_validation_results.csv`, and the model comparison figures.
- Transformer comparison: `transformer_sample_predictions.csv` if the transformer section runs successfully.
- Opinion mining and ABSA: figure files `20` to `23`, `aspect_sentiment_results.csv`, and `aspect_example_reviews.csv`.
- Error analysis: `sample_error_analysis.csv`.

The key limitation to mention is that `voted_up` is a recommendation label, not a manually annotated sentiment label. Some reviews may contain mixed written sentiment even when the player recommends or does not recommend the game.
